# The V-cycle on a real slice: what each grid level holds

A trained MGLPDS net, run on one real val slice, and the image it holds at **every
grid level** on the way back up the V-cycle (the decoder side) -- next to what a
U-Net decoder (E2E-VarNet's) holds at the same scales.

**The argument.** In MGLPDS every level carries a primal *image* $x_\ell$ that solves
a coarse version of the same reconstruction problem (FAS: the coarse variable is the
full restricted solution, not an error), so it can be scored directly against the
ground truth restricted by the net's own transfer, $R^\ell(\mathrm{gt})$. A U-Net
decoder carries 18--144 feature *channels* per scale with no defined relation to an
image; showing them at all needs a summary.

The engine is `scripts/level_figure.py` (also a CLI). The measurement is exactly
`eval_per_volume`'s for the slice -- loader, noise seed, online ESPIRiT operator,
sigma -- so the finest level reproduces `results/main/slices.csv`.

**Two ways to run**
* **Live** (cluster, GPU node, data + checkpoints): set `RUN`, `COMPARE`, `VOLUME`,
  `SLICE`. You get the arrays *and* the live objects (model, `LevelTrace`, the sparse
  codes) for section 6.
* **From a saved trace** (anywhere, numpy + matplotlib only): set `NPZ` to a file
  `save_npz` / the CLI wrote, e.g. after copying `figures/out/levels/` down.

In [ ]:
import pathlib, sys

_root = pathlib.Path.cwd()
if not (_root / "models").is_dir():
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "scripts"))

import numpy as np
import matplotlib.pyplot as plt

import importlib, os
import level_figure as lf
# Reload level_figure when the FILE changed (e.g. after a git pull) -- a plain re-import
# keeps the old module. Only then: a reload also drops its cache of loaded nets.
if getattr(lf, "_FILE_MTIME", None) != os.path.getmtime(lf.__file__):
    lf = importlib.reload(lf)
    lf._FILE_MTIME = os.path.getmtime(lf.__file__)
from visualization.image import subplot_images

# ---- EITHER a live run (cluster) ---------------------------------------------
# Change ANY of these and re-run this cell and the next: the nets are loaded ONCE per
# kernel and cached (lf.cache_info() lists them; lf.clear_cache() frees the GPU). A new
# RUN / OTHER_MG / COMPARE is loaded on first use. Section 0 compares knob settings.
# `mglpds` is mg169v6 trained with the GALERKIN coarse Gram (E . P, fine FFTs at every
# level; and gauss convs): same M=169, K=[6,[4,4,6]]. mg169v6 is the REDISCRETIZED one.
RUN     = "trained_nets/mg_recon/brain/mglpds_R8"
COARSE_OP = None            # None: as trained; "galerkin" / "rediscretize" swaps the
                            # coarse operator at inference (same weights)
TRANSFER_TAPS = None        # None: as trained (8-tap sinc); 12 or 16 implants a LONGER
                            # sinc into every transfer at inference (same weights) --
                            # applied to RUN and to OTHER_MG (section 10)
TRANSFER_FILTER = None      # None: as trained (the sinc); "box", "bilinear", ... or "none"
                            # (the sinc OFF: plain decimation down, one-sample scatter up)
                            # swaps every transfer's filter at inference (same weights) --
                            # RUN and OTHER_MG; with TRANSFER_TAPS only for "sinc"
OTHER_MG = "trained_nets/mg_recon/brain/mg169v6_R8"    # section 10: the same slice through
                                                       # a second multigrid net (None: skip)
COMPARE = "trained_nets/mg_recon/brain/varnetmaps_R8"   # None: no U-Net comparison
VOLUME  = "2000045"         # file_brain_AXT2_200_2000045; any unique part of the name
SLICE   = 4                 # SOURCE slice number; brain nets trained on slices 0-7
SIGMA   = None              # None: the run's val_noise_std (0.015)
VAL_KSPACE = "measurement_awgn"   # as results/main; "simulated" for that tree
# ---- OR a saved trace (anywhere) ---------------------------------------------
NPZ     = None              # e.g. "figures/out/levels/mglpds_R8_brain_2000045_4.npz"
SAVE_TO = "figures/out/levels"    # live mode saves the trace here (None: don't)
FLIP    = None              # None: 'ud' for knee (as the viewer), '' for brain

In [ ]:
if NPZ:
    z, meta = lf.load_npz(_root / NPZ if not pathlib.Path(NPZ).is_absolute() else NPZ)
    live = None
else:
    import os
    os.chdir(_root)                      # run dirs and the data roots are repo-relative
    z, meta, live = lf.trace_slice(RUN, VOLUME, SLICE, compare=COMPARE, sigma=SIGMA,
                                   val_kspace_type=VAL_KSPACE, coarse_op=COARSE_OP,
                                   transfer_taps=TRANSFER_TAPS, transfer_filter=TRANSFER_FILTER)
    if SAVE_TO:
        print("saved", lf.save_npz(z, meta, SAVE_TO))
    print("cached nets:\n" + lf.cache_info())

FLIP = lf.default_flip(meta) if FLIP is None else FLIP
F = lambda a: lf._flip(a, FLIP) if FLIP else a          # display orientation only

print(f"{meta['run']}  M={meta['M']}  K={meta['K']}  {meta['n_params']:,} params   "
      f"{meta['volume']} slice {meta['slice']}  R={meta['R']}  sigma={meta['sigma']:g}")
print(f"coarse operator: {meta.get('coarse_op', '?')}"
      + (f"  (SWAPPED at inference; trained with {meta.get('coarse_op_trained')})"
         if meta.get("coarse_op_swapped") else "  (as trained)"))
print(f"transfer filter: {meta.get('transfer_taps', '?')}-tap sinc"
      + (f"  (IMPLANTED at inference; trained with {meta.get('transfer_taps_trained')})"
         if meta.get("transfer_taps_swapped") else "  (as trained)"))
print(f"transfer filter: {meta.get('transfer_filter', '?')}"
      + (f"  (SWAPPED at inference; trained with {meta.get('transfer_filter_trained')})"
         if meta.get("transfer_filter_swapped") else "  (as trained)"))
print(f"DC recovered {complex(*meta['dc']):.4g}, misfit {meta['scale_misfit']:.1e} "
      f"(should be ~1e-7: the finest level + DC IS the output)")
for l, shape, errs in lf.level_table(z, meta):
    print(f"  level {l}  {shape[0]:>4}x{shape[1]:<4}  NRMSE vs R^l(gt) by V-cycle: "
          + "  ".join(f"{100 * e:5.1f}%" for e in errs))

## 0. Knob sandbox

`RUN` re-traced on the same slice under several knob settings -- one forward each, the
net stays loaded -- against one table: the output's NRMSE and each level's (last
V-cycle by default). Knobs per row: `coarse_op`, `transfer_filter`, `transfer_taps`,
`sigma`. Coarse-level NRMSEs are scored against $R^\ell(\mathrm{gt})$ through *that
row's* transfer, so across transfer changes only the output column compares directly.
The main trace (`z`, `meta`, `live`) is left as it is.

In [ ]:
SANDBOX_RUN = RUN            # or OTHER_MG
SANDBOX = [                  # (label, knobs)
    ("as trained",     {}),
    ("galerkin",       dict(coarse_op="galerkin")),
    ("rediscretize",   dict(coarse_op="rediscretize")),
    ("box transfer",   dict(transfer_filter="box")),
    ("no filter",      dict(transfer_filter="none")),
    ("16-tap sinc",    dict(transfer_taps=16)),
]
O_SB   = -1                  # which V-cycle the per-level columns read
SHOW_V = False               # also draw each row's V-cycle (section 7's figure)

if NPZ:
    print("live mode only (NPZ is set)")
else:
    rows = []
    for label, kn in SANDBOX:
        kn = dict(kn)
        s = kn.pop("sigma", SIGMA)
        try:
            zk, mk, _ = lf.trace_slice(SANDBOX_RUN, VOLUME, SLICE, sigma=s,
                                       val_kspace_type=VAL_KSPACE, **kn)
        except ValueError as e:                 # e.g. taps on a non-sinc net
            print(f"{label}: skipped -- {e}")
            continue
        per = {l: errs[O_SB] for l, _, errs in lf.level_table(zk, mk)}
        rows.append((label, lf.nrmse(zk["recon"], zk["gt"]), per))
        if SHOW_V:
            fig = lf.fig_vcycle(zk, mk, outer=O_SB, flip=FLIP); plt.suptitle(label); plt.show()
    levels = sorted(rows[0][2], reverse=True) if rows else []
    print(f"\n{os.path.basename(SANDBOX_RUN)}  {VOLUME} slice {SLICE}  (NRMSE, whole frame; "
          f"levels after V-cycle {O_SB if O_SB >= 0 else len(meta['outers']) + O_SB + 1})")
    print(f"{'':<16}{'output':>10}" + "".join(f"{'L' + str(l):>9}" for l in levels))
    for label, final, per in rows:
        print(f"{label:<16}{100 * final:9.3f}%" + "".join(f"{100 * per[l]:8.1f}%" for l in levels))

## 1. The decoder side

One column per level, coarsest $\to$ finest, after the last V-cycle has come back up
through it. Rows: $x_\ell$; $R^\ell(\mathrm{gt})$; the residual (windowed at
vmax / `resid_gain`, as the viewer); and $|\Delta x|$, what the level below handed up
(on the grid it lives on; the finest level has none). Every panel is at its own grid
size -- nothing is resampled. `outer=` picks another V-cycle (0-based, -1 = last).

In [ ]:
fig = lf.fig_levels(z, meta, outer=-1, flip=FLIP)
plt.show()

## 2. Across the outer V-cycles

Each level after each of the outer cycles; panel titles are NRMSE vs $R^\ell(\mathrm{gt})$.
The coarse levels settle first -- the low frequencies are what a coarse grid can fix --
and the fine level keeps sharpening.

In [ ]:
fig = lf.fig_outers(z, meta, flip=FLIP)
plt.show()

# the same numbers as curves
fig, ax = plt.subplots(figsize=(5, 3.4))
for l, shape, errs in lf.level_table(z, meta):
    ax.plot(np.arange(1, len(errs) + 1), [100 * e for e in errs], marker="o",
            label=f"level {l} ({shape[0]}x{shape[1]})")
ax.set_xlabel("outer V-cycle"); ax.set_ylabel("NRMSE vs R^l(gt)  (%)")
ax.legend(frameon=False); ax.grid(alpha=.3)
plt.show()

## 3. The U-Net decoder at its scales

The compared VarNet's last cascade: its U-Net's decoder blocks, coarse $\to$ fine.
Channel RMS and the first principal component (and how much of the channel variance
it explains) -- a summary is the only way to look at them at all.

In [ ]:
fig = lf.fig_unet(z, meta, flip=FLIP)
if fig is None:
    print("no comparison in this trace (COMPARE=None)")
else:
    plt.show()

## 4. Do it yourself

Everything is in `z` (numpy) and `meta`. Accessors below; `F()` applies the display
flip. `subplot_images` (visualization/image.py) shares one window across panels
unless told otherwise -- pass `share_window=False` for panels on different scales.

In [ ]:
LEVELS, OUTERS = meta["levels"], meta["outers"]
X  = lambda l, o=-1: z[f"mg/x/o{OUTERS[o]}/l{l}"]                 # complex, image units
GT = lambda l: z[f"mg/gt/l{l}"]                                   # R^l(gt)
D  = lambda l, o=-1: z.get(f"mg/delta/o{OUTERS[o]}/l{l}")         # handed up (l >= 1)
ORGAN = lambda l: z[f"mg/organ/l{l}"]                             # R^l(organ mask)
FEAT = sorted((k for k in z if k.startswith("vn/feat/")), key=lambda k: int(k.rsplit("/", 1)[1]))

print("levels", LEVELS, "outers", OUTERS, "| U-Net scales", [z[k].shape for k in FEAT])

# example: the final reconstruction next to every coarse level, each on its own window
subplot_images([F(np.abs(X(l))) for l in sorted(LEVELS, reverse=True)] + [F(np.abs(GT(0)))],
               titles=[f"level {l}" for l in sorted(LEVELS, reverse=True)] + ["ground truth"],
               share_window=False)

In [ ]:
# NRMSE inside the organ (the region the metrics score) vs the whole frame, per level
for l in sorted(LEVELS, reverse=True):
    m = ORGAN(l) > 0.5
    x, g = np.abs(X(l)), np.abs(GT(l))
    inside = np.linalg.norm((x - g)[m]) / np.linalg.norm(g[m])
    print(f"level {l}: whole frame {100 * lf.nrmse(X(l), GT(l)):5.1f}%   organ {100 * inside:5.1f}%")

In [ ]:
# browse individual U-Net channels at one scale (0 = coarsest decoder block)
if FEAT:
    K = 0
    f = z[FEAT[K]].astype(np.float32)
    top = np.argsort(-(f ** 2).mean((1, 2)))[:16]            # the 16 most energetic
    subplot_images([F(f[c]) for c in top], titles=[f"ch {c}" for c in top],
                   ncols=8, cmap="RdBu_r", symmetric=True, share_window=False,
                   suptitle=f"U-Net decoder block {K}: {f.shape[0]} channels at {f.shape[1]}x{f.shape[2]}")

## 5. Zoom: the same region at every level

A box given in FINEST-grid pixels (displayed orientation), mapped onto each coarser
grid by the factor 2 per level -- how much detail each level actually carries there.

In [ ]:
BOX = None    # (row, col, height, width) on the finest displayed grid; None: centre quarter
H0, W0 = X(0).shape
r, c, h, w = BOX or (H0 // 2 - H0 // 8, W0 // 2 - W0 // 8, H0 // 4, W0 // 4)
ims, titles = [], []
for l in sorted(LEVELS, reverse=True):
    s = 2 ** l
    for name, a in (("x", X(l)), ("gt", GT(l))):
        A = F(np.abs(a))
        ims.append(A[r // s:(r + h) // s, c // s:(c + w) // s]); titles.append(f"{name} level {l}")
subplot_images(ims, titles=titles, ncols=2, share_window=True,
               suptitle=f"rows {r}:{r + h}, cols {c}:{c + w} of the finest grid")

## 6. Live only: the sparse codes, the FAS corrections, the model

`live["mg"]["trace"]` is the full `LevelTrace` (models/level_trace.py): per
(outer, level) the incoming / outgoing iterates `x`, `z`, `x_out`, `z_out`, what the
level below receives (`x_c`, `z_c`), the FAS corrections (`pi_x`, `pi_z`) and
`coarse_delta_*`. These are on the NETWORK's scale (DC removed), and under the planar
state they are real with [re; im] stacked on the channel axis -- `as_complex` undoes it.

`z` is the dual / code variable: M channels per level, on the LATENT grid -- the
image grid divided by the conv stride `s` (2), so half each level's image size.
Unlike a U-Net's channels each one is tied to a dictionary atom
(`visualization.filters.effective_atoms(model)`).

In [ ]:
if live is None:
    print("loaded from NPZ: no live objects -- run in live mode for this section")
else:
    import torch
    tr, model = live["mg"]["trace"], live["mg"]["model"]
    print(tr.summary())

    def as_complex(t):
        # the planar state stores [re; im] stacked on channels
        if t is None or t.is_complex():
            return t
        h = t.shape[1] // 2
        return torch.complex(t[:, :h], t[:, h:])

    # code energy per channel at each level, last V-cycle
    for l in sorted(LEVELS, reverse=True):
        zc = as_complex(tr.get(-1, l, "z_out"))
        if zc is None:
            continue
        e = zc.abs().pow(2).mean((0, 2, 3)).cpu().numpy()
        print(f"level {l}: z {tuple(zc.shape)}  active channels (>1% of max energy): "
              f"{int((e > 0.01 * e.max()).sum())}/{e.size}")

In [ ]:
# the |z| maps of the most energetic code channels at one level -- each channel is an atom's coefficient map
if live is not None:
    L = max(LEVELS)                      # try 0 for the finest
    zc = as_complex(tr.get(-1, L, "z_out"))[0].abs().cpu().numpy()
    top = np.argsort(-(zc ** 2).mean((1, 2)))[:16]
    subplot_images([F(zc[m]) for m in top], titles=[f"atom {m}" for m in top], ncols=8,
                   share_window=False, suptitle=f"|z| at level {L}: the 16 most active atoms")

## 7. Every branch of one V-cycle

The V-cycle drawn as a V: a row per level (finest on top), time left to right --
enter, pre-smooth, restrict down, ..., coarsest solve, ..., coarse correction,
post-smooth. Yellow: NRMSE vs $R^\ell(\mathrm{gt})$. `show="resid"` draws
$|x_\ell - R^\ell \mathrm{gt}|$ instead.

In [ ]:
OUTER_LIST = [0, -1]         # which V-cycles: e.g. [0, -1], or list(range(len(meta["outers"])))
SHOW_LIST  = ["x"]           # "x" = the iterate, "resid" = |x_l - R^l gt| (window vmax / 5)
CROP, DPI, PANEL = True, 130, 2.0

if not any(k.startswith("mg/xpre/") for k in z):
    print("this trace predates the per-branch hooks: re-run the trace cell (live mode)")
else:
    for o in OUTER_LIST:
        for show in SHOW_LIST:
            fig = lf.fig_vcycle(z, meta, outer=o, show=show, flip=FLIP, crop=CROP,
                                panel=PANEL, dpi=DPI)
            plt.show()

## 8. Every level after every V-cycle

One row per outer cycle, one column group per level (coarsest $\to$ finest): the
iterate, its residual and the correction it hands up. Windows fixed per level across
rows. `TRUE_SIZE` draws coarse levels physically smaller.

In [ ]:
SHOW       = ("x", "resid", "delta")  # any of: "x", "gt", "resid", "delta"
TRUE_SIZE  = True
CROP       = True
CROP_PAD   = 8
INTERP     = "antialiased"            # "nearest" aliases when a panel is smaller than the image
RESID_GAIN = 5.0
PANEL      = 3.0
DPI        = 150

order = sorted(LEVELS, reverse=True)
r0, r1, c0, c1 = lf.crop_box(z, meta, FLIP, CROP_PAD) if CROP else (0, None, 0, None)
cut = lambda a, l: F(a)[r0 // 2 ** l:(None if r1 is None else r1 // 2 ** l),
                        c0 // 2 ** l:(None if c1 is None else c1 // 2 ** l)]
labels = {"x": "x", "gt": "gt", "resid": "|err|", "delta": "|dx| up"}
cols = [(l, k) for l in order for k in SHOW if not (k == "delta" and l == min(order))]
vmax = {l: float(np.percentile(np.abs(cut(GT(l), l)), 99.5)) or 1.0 for l in order}
dmax = {}
for l in order:
    ds = [np.abs(cut(D(l, o), l)) for o in range(len(OUTERS)) if D(l, o) is not None]
    dmax[l] = float(np.percentile(np.concatenate([d.ravel() for d in ds]), 99.5)) if ds else 1.0
ratios = [(1 / 2 ** l if TRUE_SIZE else 1.0) for l, _ in cols]
g0 = cut(GT(0), 0)
fig, ax = plt.subplots(len(OUTERS), len(cols), squeeze=False, dpi=DPI,
                       figsize=(PANEL * sum(ratios) + 1.0,
                                PANEL * g0.shape[0] / g0.shape[1] * len(OUTERS) + 0.8),
                       gridspec_kw=dict(width_ratios=ratios, wspace=0.04, hspace=0.12))
for i in range(len(OUTERS)):
    for j, (l, k) in enumerate(cols):
        a = ax[i][j]
        a.set_xticks([]); a.set_yticks([])
        x, g = X(l, i), GT(l)
        if k == "x":
            im, cm, hi = np.abs(x), "gray", vmax[l]
        elif k == "gt":
            im, cm, hi = np.abs(g), "gray", vmax[l]
        elif k == "resid":
            im, cm, hi = np.abs(np.abs(x) - np.abs(g)), "inferno", vmax[l] / RESID_GAIN
        else:
            d = D(l, i)
            if d is None:
                a.set_frame_on(False)
                continue
            im, cm, hi = np.abs(d), "magma", dmax[l]
        a.imshow(cut(im, l), cmap=cm, vmin=0, vmax=hi, interpolation=INTERP)
        if k == "x":
            a.text(0.03, 0.97, f"{100 * lf.nrmse(x, g):.1f}%", transform=a.transAxes,
                   ha="left", va="top", fontsize=7, color="yellow")
        if i == 0:
            a.set_title(f"L{l} {labels[k]}", fontsize=8)
    ax[i][0].set_ylabel(f"V-cycle {OUTERS[i] + 1}", fontsize=9)
grids = "  ".join(f"L{l} {GT(l).shape[0]}x{GT(l).shape[1]}" for l in order)
fig.suptitle(f"{meta['run']} ({meta.get('coarse_op', '?')} coarse op)  {meta['volume']} slice "
             f"{meta['slice']}: every level after each V-cycle  ({grids})\nx = x_l, gt = R^l(gt), "
             f"|err| = |x_l - R^l gt|, |dx| up = correction handed up; yellow = NRMSE vs R^l(gt)",
             fontsize=9)
plt.show()

# is each V-cycle's input the previous one's output, and does the error fall monotonically?
for i in range(1, len(OUTERS)):
    a, b = z[f"mg/x/o{OUTERS[i-1]}/l0"], z.get(f"mg/xin/o{OUTERS[i]}/l0")
    if b is not None:
        print(f"V-cycle {i} out -> {i + 1} in: rel. diff {np.linalg.norm(a - b) / np.linalg.norm(b):.1e}"
              f"   NRMSE out of V-cycle {i}: {100 * lf.nrmse(a, GT(0)):.1f}%")

## 9. Is the coarsest level solving its problem?

* **A** -- where each stage's error sits: a high NRMSE that survives low-passing is a
  wrong *image*; error mostly above the cut is band-edge content, which prolongation
  filters out before the finer level sees it.
* **B** (live) -- how much each level takes from the one below: $\alpha_x$ and
  $\|\alpha_x P(w_c - x_c)\| / \|x_\mathrm{pre}\|$.
* **C** (live, `RUN_SWAP = True`) -- the same checkpoint with the OTHER coarse operator
  swapped in at inference (rediscretize $\leftrightarrow$ galerkin). One more forward.

In [ ]:
CUT = 0.5                     # "low" = below this fraction of the coarse grid's Nyquist
O = -1                        # which outer V-cycle
RUN_SWAP = False

def lowpass(a, cut=CUT):
    Fa = np.fft.fftshift(np.fft.fft2(a))
    h, w = a.shape
    ky = np.abs(np.arange(h) - h // 2) / (h / 2)
    kx = np.abs(np.arange(w) - w // 2) / (w / 2)
    keep = (ky[:, None] <= cut) & (kx[None, :] <= cut)
    return np.real(np.fft.ifft2(np.fft.ifftshift(Fa * keep))), keep

def high_share(err, keep):
    P = np.abs(np.fft.fftshift(np.fft.fft2(err))) ** 2
    return float(P[~keep].sum() / P.sum())

def stage_table(z, meta, O=-1):
    o = meta["outers"][O]
    print(f"   {'stage':<28}{'NRMSE':>8}{'NRMSE low-passed':>18}{'error energy above cut':>25}")
    for l in sorted(meta["levels"], reverse=True):
        g = np.abs(z[f"mg/gt/l{l}"])
        for key, name in (("xin", "enters / restricted down"), ("xpre", "pre-smoothed"),
                          ("xcorr", "+ coarse correction"), ("x", "solve / post-smoothed")):
            k = f"mg/{key}/o{o}/l{l}"
            if k not in z:
                continue
            x = np.abs(z[k])
            xl, keep = lowpass(x); gl, _ = lowpass(g)
            print(f"   L{l} {name:<25}{100 * lf.nrmse(x, g):7.1f}%"
                  f"{100 * np.linalg.norm(xl - gl) / np.linalg.norm(gl):17.1f}%"
                  f"{100 * high_share(x - g, keep):24.0f}%")

def _takes_table(live, O=-1):
    import torch
    from models.level_trace import _level_map
    from models.mg_lpds import PDVCycle
    from operators.resample import prolong
    tr, model = live["mg"]["trace"], live["mg"]["model"]
    def as_complex(t):
        if t is None or t.is_complex():
            return t
        h = t.shape[1] // 2
        return torch.complex(t[:, :h], t[:, h:])
    o = tr.outers[O]
    lvl = _level_map(model, PDVCycle)[0]
    tops = [m for m in model.modules() if isinstance(m, PDVCycle) and lvl[id(m)] == 0]
    v, l = tops[tr.outers.index(o)], 0
    while isinstance(v, PDVCycle):
        with torch.no_grad():
            dev = next(v.parameters()).device
            a = complex(v.alpha_x(torch.ones(1, 1, 1, 1, dtype=torch.complex64, device=dev)).flatten()[0])
            d = as_complex(tr.get(o, l, "coarse_delta_x"))
            corr = as_complex(v.alpha_x(prolong(d, filter=v.transfer_filter)))
            pre = as_complex(tr.get(o, l, "x_pre"))
        print(f"   L{l + 1} -> L{l}: alpha_x = {a.real:+.3g}{a.imag:+.3g}j   "
              f"|w_c - x_c| / |x_c| = {float(d.norm() / as_complex(tr.get(o, l, 'x_c')).norm()):.2f}   "
              f"|alpha P(w_c - x_c)| / |x_pre| = {float(corr.norm() / pre.norm()):.3f}")
        v, l = v.mglayer, l + 1

def takes_table(live, O=-1):
    # the nets are cached and re-used by later traces: put this one back in the
    # state its trace ran it with (coarse op, transfer filter, sinc length)
    with lf.as_traced(live):
        _takes_table(live, O)

print(f"A. {meta['run']} ({meta.get('coarse_op', '?')}), V-cycle {OUTERS[O] + 1}")
stage_table(z, meta, O)
if live is not None:
    print(f"\nB. what each level takes from the one below")
    takes_table(live, O)
    if RUN_SWAP and not NPZ:
        other_op = "galerkin" if meta.get("coarse_op") == "rediscretize" else "rediscretize"
        zs, ms, ls = lf.trace_slice(RUN, VOLUME, SLICE, sigma=SIGMA, val_kspace_type=VAL_KSPACE,
                                    coarse_op=other_op, transfer_taps=TRANSFER_TAPS,
                                    transfer_filter=TRANSFER_FILTER)
        print(f"\nC. same checkpoint with the {other_op} coarse operator (trained: "
              f"{ms.get('coarse_op_trained')})")
        stage_table(zs, ms, O)
        takes_table(ls, O)
        fig = lf.fig_vcycle(zs, ms, outer=O, flip=FLIP); plt.show()

## 10. A second multigrid net on the same slice

`OTHER_MG` (default: the REDISCRETIZED mg169v6) traced on exactly the same
measurement, next to `RUN` (default: the GALERKIN-trained mglpds) -- same width and
depth, so the coarse operator is the main difference. Live only. The per-level tables
of section 9 and the V figure, for both.

In [ ]:
if live is None or not OTHER_MG:
    print("live mode with OTHER_MG set only")
else:
    z2, meta2, live2 = lf.trace_slice(OTHER_MG, VOLUME, SLICE, sigma=SIGMA,
                                      val_kspace_type=VAL_KSPACE, transfer_taps=TRANSFER_TAPS,
                                      transfer_filter=TRANSFER_FILTER)
    if SAVE_TO:
        print("saved", lf.save_npz(z2, meta2, SAVE_TO))
    for zz, mm, ll in ((z, meta, live), (z2, meta2, live2)):
        print(f"\n=== {mm['run']}  ({mm.get('coarse_op')} coarse op, {mm['n_params']:,} params)")
        for l, shape, errs in lf.level_table(zz, mm):
            print(f"  level {l}  {shape[0]:>4}x{shape[1]:<4}  NRMSE vs R^l(gt) by V-cycle: "
                  + "  ".join(f"{100 * e:5.1f}%" for e in errs))
        stage_table(zz, mm, O)
        takes_table(ll, O)
    for zz, mm in ((z, meta), (z2, meta2)):
        fig = lf.fig_vcycle(zz, mm, outer=O, flip=FLIP); plt.show()

## 11. Where do the stripes come from?

The coarse operator is ruled out: the stripes survive a swap to the Galerkin Gram, and
`mglpds`, trained with Galerkin, shows them too. The next suspect is the **stride-2
convolutions** in every LPDS layer (`models/lpds.py`: a stride-2 analysis `Conv2d` and a
stride-2 synthesis `ConvTranspose2d`, P = 7). A 7-tap kernel at stride 2 overlaps
unevenly -- alternate output pixels get 4 taps and 3 -- which is the classic source of
period-2 grid artifacts.

A period-2 pattern along an axis puts its energy at that axis' Nyquist frequency, so the
measure is the share of an image's energy (mean removed) near Nyquist, along each axis of
its own grid. **rows** = along axis 0 (horizontal stripes; for these fastMRI arrays axis 0
is readout, the long oversampled dimension), **cols** = along axis 1 (vertical stripes,
phase-encode). The ground truth at each level is the baseline.

* **A** -- every stage of the traced V-cycle (from the trace above, no forward pass).
* **B** -- every LPDS layer, in execution order, of `RUN` and of a flat LPDSNet
  (`FLAT_RUN`: the same stride-2 layers, no coarse levels). If the flat net's
  intermediate iterates stripe too, the stride is the source, not the V-cycle.

In [ ]:
FRAC = 0.8     # "near Nyquist" = at or above this fraction of each grid's Nyquist
O_ST = -1      # which V-cycle
o = OUTERS[O_ST]
print(f"A. {meta['run']}, V-cycle {o + 1}: share of energy near Nyquist "
      f"(rows = horizontal stripes, cols = vertical)")
print(f"   {'stage':<30}{'rows':>9}{'cols':>9}")
for l in sorted(LEVELS, reverse=True):
    r, c = lf.nyquist_share(GT(l), FRAC)
    print(f"   L{l} {'ground truth R^l(gt)':<27}{100 * r:8.2f}%{100 * c:8.2f}%")
    for key, name in (("xin", "enters / restricted down"), ("xpre", "pre-smoothed"),
                      ("xcorr", "+ coarse correction"), ("x", "solve / post-smoothed")):
        k = f"mg/{key}/o{o}/l{l}"
        if k in z:
            r, c = lf.nyquist_share(z[k], FRAC)
            print(f"   L{l} {name:<27}{100 * r:8.2f}%{100 * c:8.2f}%")

In [ ]:
FLAT_RUN = "trained_nets/mg_recon/brain/lpdsnet_R8"   # stride-2 layers, no coarse levels; None: skip
GALLERY  = [0, 4, 9, 19, -1]                          # flat-net layers to show (0-based; -1 = last)

if NPZ:
    print("live mode only (NPZ is set)")
else:
    runs = [RUN] + ([FLAT_RUN] if FLAT_RUN else [])
    traced = {}
    fig, axes = plt.subplots(1, len(runs), figsize=(7 * len(runs), 3.8), squeeze=False)
    for ax, run in zip(axes[0], runs):
        knobs = dict(coarse_op=COARSE_OP, transfer_filter=TRANSFER_FILTER) if run == RUN else {}
        layers, gts, lm = lf.trace_layers(run, VOLUME, SLICE, sigma=SIGMA,
                                          val_kspace_type=VAL_KSPACE, **knobs)
        traced[run] = (layers, gts, lm)
        shapes = sorted({L["shape"] for L in layers}, key=lambda s: -s[0] * s[1])
        lvl = {s: i for i, s in enumerate(shapes)}
        ks = [L["k"] for L in layers]
        sh = [lf.nyquist_share(L["x"], FRAC) for L in layers]
        ax.plot(ks, [100 * r for r, _ in sh], color="C0", lw=1.4, label="rows (horizontal stripes)")
        ax.plot(ks, [100 * c for _, c in sh], color="C1", lw=1.4, label="cols (vertical stripes)")
        for s, i in lvl.items():                       # which grid each layer ran on
            sel = [j for j, L in enumerate(layers) if L["shape"] == s]
            ax.scatter([ks[j] for j in sel], [100 * sh[j][0] for j in sel], s=10,
                       color=f"C{2 + i}", zorder=3, label=f"level {i} {s[0]}x{s[1]}")
            if s in gts:
                gr, _ = lf.nyquist_share(gts[s], FRAC)
                ax.axhline(100 * gr, color=f"C{2 + i}", ls=":", lw=1)
        ax.set_xlabel("LPDS layer (execution order)")
        ax.set_ylabel(f"energy at >= {FRAC:g} Nyquist (%)")
        ax.set_title(f"{lm['run']}: {lm['n_layers']} layers  (dotted: ground truth at that level)",
                     fontsize=10)
        ax.grid(alpha=0.3); ax.legend(fontsize=8, frameon=False)
    plt.tight_layout(); plt.show()

    if FLAT_RUN:
        layers, gts, lm = traced[FLAT_RUN]
        pick = [layers[i] for i in GALLERY]
        g0 = gts[layers[-1]["shape"]]
        subplot_images([F(np.abs(L["x"])) for L in pick] + [F(np.abs(g0))],
                       titles=[f"layer {L['k'] + 1}  rows {100 * lf.nyquist_share(L['x'], FRAC)[0]:.2f}%"
                               for L in pick] + ["ground truth"],
                       share_window=False, suptitle=f"{lm['run']}: iterates through the stack")

## 12. Is it the stride-2 convs? A weight check and an intervention

Section 11 shows *where* period-2 energy appears; this tests *whether the stride-2 convs put
it there*.

A stride-2 conv with an odd kernel is four filters, one per output parity: with P = 7 an
even output row gets taps {1, 3, 5} and an odd one {0, 2, 4, 6}. So the synthesis maps a
latent DC (or any slowly varying code) to its DC gain $W(0,0)$ **plus** a period-2 pattern
$W(\pi,0)(-1)^{\text{row}} + W(0,\pi)(-1)^{\text{col}} + W(\pi,\pi)(-1)^{\text{row}+\text{col}}$,
where $W(\pi,0) = \sum_{ij}(-1)^i w_{ij}$ = (even-row taps) $-$ (odd-row taps). The analysis
conv is its adjoint: image content at those frequencies *aliases* onto the latent DC with the
same gains. Nothing in training pins them to zero.

* **12a -- the weights.** $\|W(\pi,0)\|/\|W(0,0)\|$ (**rows**: horizontal stripes),
  $\|W(0,\pi)\|/\|W(0,0)\|$ (**cols**) and the diagonal, for every analysis / synthesis conv,
  by grid level (Frobenius norms over the channel pairs). **~1 is a random kernel's level
  (the init); 0 is no leak.** No data, no forward pass. Compare the multigrid net's levels with
  each other and with a flat LPDSNet (`FLAT_RUN12`: same anatomy and R as `RUN`).
* **12b / 12c -- the intervention.** Inside `lf.equalized_convs` every selected conv has its
  parity classes balanced -- the smallest change that zeroes $W(\pi,\cdot)$ / $W(\cdot,\pi)$
  while keeping each kernel's DC gain -- then the trained weights come back. Same idea as the
  transfer swaps: same net, one property removed. **If the stripes vanish and NRMSE barely
  moves, the convs' leak is the cause.** If the stripes stay, it is not. If NRMSE blows up,
  the net *uses* the leak and the test is inconclusive -- narrow it with `EQ_AXES` (only the
  stripe axis), `EQ_CONVS` (synthesis only) or `EQ_LEVELS` (coarse levels only).

In [ ]:
# 12a. How much does each trained conv leak? (weights only)
_r = RUN.rstrip("/")
FLAT_RUN12 = os.path.join(os.path.dirname(_r), "lpdsnet_" + _r.rsplit("_", 1)[-1])  # None: skip
FRAC = globals().get("FRAC", 0.8)

if NPZ:
    print("live mode only (NPZ is set)")
else:
    runs12 = [RUN] + ([FLAT_RUN12] if FLAT_RUN12 else [])
    leak = {run: lf.leak_table(lf.cached_model(run)) for run in runs12}
    for run, rows in leak.items():
        print(f"\n{os.path.basename(run)}: |W(pi)| / |W(0)|, mean (max) over the convs at each level"
              f"   [~1 = a random kernel, 0 = no leak]")
        print(f"   {'level':<7}{'conv':<11}{'n':>4}{'rows':>15}{'cols':>15}{'diag':>15}")
        for l, conv, n, mean, mx in lf.leak_summary(rows):
            print(f"   L{l:<6}{conv:<11}{n:>4}" + "".join(f"{m:8.3f} ({x:4.2f})" for m, x in zip(mean, mx)))

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
    for ax, conv in zip(axes, ("analysis", "synthesis")):
        for run, ls in zip(runs12, ("-", "--")):
            S = [s for s in lf.leak_summary(leak[run]) if s[1] == conv]
            L = [s[0] for s in S]
            ax.plot(L, [s[3][0] for s in S], "o" + ls, color="C0", label=f"{os.path.basename(run)}: rows")
            ax.plot(L, [s[3][1] for s in S], "s" + ls, color="C1", label=f"{os.path.basename(run)}: cols")
        ax.axhline(1.0, color="0.6", ls=":", lw=1)
        ax.set_title(f"{conv} convs (mean over the level)", fontsize=10)
        ax.set_xlabel("grid level (0 = finest)")
        ax.set_xticks(sorted({s[0] for s in lf.leak_summary(leak[RUN])}))
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("|W(pi)| / |W(0)|")
    axes[1].legend(fontsize=8, frameon=False)
    plt.tight_layout(); plt.show()

In [ ]:
# 12b. The intervention on RUN: output and every level, trained vs leak-free convs
EQ_CONVS  = ("analysis", "synthesis")   # which convs lose their period-2 leak
EQ_LEVELS = None                        # None: every level; e.g. [1, 2] = the coarse levels only
EQ_AXES   = "both"                      # "both", "rows" (horizontal stripes only) or "cols"
DIFF_GAIN = 5.0                         # |equalized - trained| shown x this
FRAC = globals().get("FRAC", 0.8)

if NPZ:
    print("live mode only (NPZ is set)")
else:
    kn = dict(sigma=SIGMA, val_kspace_type=VAL_KSPACE, coarse_op=COARSE_OP,
              transfer_filter=TRANSFER_FILTER)
    z_off, m_off, _ = lf.trace_slice(RUN, VOLUME, SLICE, **kn)
    with lf.equalized_convs(lf.cached_model(RUN), EQ_CONVS, EQ_LEVELS, EQ_AXES) as n_eq:
        z_on, m_on, _ = lf.trace_slice(RUN, VOLUME, SLICE, **kn)
    print(f"{m_off['run']}: {n_eq} convs equalized  (axes={EQ_AXES}, convs={EQ_CONVS}, "
          f"levels={EQ_LEVELS or 'all'}); weights restored")
    print(f"   {'':<30}{'NRMSE':>9}{'rows':>9}{'cols':>9}   rows/cols: energy at >= {FRAC:g} Nyquist")

    def _line(name, x, g=None):
        r, c = lf.nyquist_share(x, FRAC)
        e = f"{100 * lf.nrmse(x, g):8.2f}%" if g is not None else " " * 9
        print(f"   {name:<30}{e}{100 * r:8.2f}%{100 * c:8.2f}%")

    _line("output: ground truth", z_off["gt"])
    _line("output: trained convs", z_off["recon"], z_off["gt"])
    _line("output: equalized", z_on["recon"], z_on["gt"])
    o = m_off["outers"][-1]
    print(f"\n   V-cycle {o + 1}, every stage (as section 11A): near-Nyquist share, trained -> equalized")
    print(f"   {'stage':<30}{'rows':>20}{'cols':>20}")
    for l in sorted(m_off["levels"], reverse=True):
        gr, gc = lf.nyquist_share(z_off[f"mg/gt/l{l}"], FRAC)
        print(f"   L{l} {'ground truth R^l(gt)':<27}{100 * gr:19.2f}%{100 * gc:19.2f}%")
        for key, name in (("xin", "enters / restricted down"), ("xpre", "pre-smoothed"),
                          ("xcorr", "+ coarse correction"), ("x", "solve / post-smoothed")):
            k = f"mg/{key}/o{o}/l{l}"
            if k in z_off:
                (ar, ac), (br, bc) = lf.nyquist_share(z_off[k], FRAC), lf.nyquist_share(z_on[k], FRAC)
                print(f"   L{l} {name:<27}{100 * ar:9.2f}% ->{100 * br:6.2f}%{100 * ac:10.2f}% ->{100 * bc:6.2f}%")

    r0, r1, c0, c1 = lf.crop_box(z_off, m_off, flip=FLIP)
    top = max(m_off["levels"])
    crop = lambda a, l=0: F(a)[r0 // 2 ** l:r1 // 2 ** l, c0 // 2 ** l:c1 // 2 ** l]  # noqa: E731
    dif = np.abs(np.abs(z_on["recon"]) - np.abs(z_off["recon"])) * DIFF_GAIN
    subplot_images([crop(np.abs(z_off["recon"])), crop(np.abs(z_on["recon"])), crop(dif),
                    crop(np.abs(z_off[f"mg/x/o{o}/l{top}"]), top),
                    crop(np.abs(z_on[f"mg/x/o{o}/l{top}"]), top)],
                   titles=["output, trained", "output, equalized",
                           f"|difference| x {DIFF_GAIN:g}", f"L{top}, trained", f"L{top}, equalized"],
                   share_window=False, suptitle=f"{m_off['run']}: stride-2 leak removed ({EQ_AXES})")

In [ ]:
# 12c. Per LPDS layer, trained vs equalized: RUN and the flat FLAT_RUN12
import contextlib

if NPZ:
    print("live mode only (NPZ is set)")
else:
    runs12 = [RUN] + ([FLAT_RUN12] if FLAT_RUN12 else [])
    fig, axes = plt.subplots(1, len(runs12), figsize=(7 * len(runs12), 3.8), squeeze=False)
    for ax, run in zip(axes[0], runs12):
        knobs = dict(coarse_op=COARSE_OP, transfer_filter=TRANSFER_FILTER) if run == RUN else {}
        res = {}
        for tag in ("trained", "equalized"):
            ctx = (lf.equalized_convs(lf.cached_model(run), EQ_CONVS, EQ_LEVELS, EQ_AXES)
                   if tag == "equalized" else contextlib.nullcontext(0))
            with ctx as n:
                res[tag] = lf.trace_layers(run, VOLUME, SLICE, sigma=SIGMA,
                                           val_kspace_type=VAL_KSPACE, **knobs)
            if tag == "equalized":
                n_run = n
        for tag, ls in (("trained", "-"), ("equalized", "--")):
            layers, gts, lm = res[tag]
            sh = [lf.nyquist_share(L["x"], FRAC) for L in layers]
            ks = [L["k"] for L in layers]
            ax.plot(ks, [100 * r for r, _ in sh], ls, color="C0", lw=1.3, label=f"rows, {tag}")
            ax.plot(ks, [100 * c for _, c in sh], ls, color="C1", lw=1.3, label=f"cols, {tag}")
            if tag == "trained":
                final = layers[-1]
                g = gts[final["shape"]]
                gr, gc = lf.nyquist_share(g, FRAC)
                ax.axhline(100 * gr, color="C0", ls=":", lw=1)
                ax.axhline(100 * gc, color="C1", ls=":", lw=1)
            print(f"{lm['run']:<16} {tag:<10} last layer NRMSE {100 * lf.nrmse(layers[-1]['x'], gts[layers[-1]['shape']]):6.2f}%"
                  f"  (embedded frame)")
        ax.set_xlabel("LPDS layer (execution order)")
        ax.set_ylabel(f"energy at >= {FRAC:g} Nyquist (%)")
        ax.set_title(f"{lm['run']}: {n_run} convs equalized (dotted: ground truth, finest grid)",
                     fontsize=10)
        ax.grid(alpha=0.3); ax.legend(fontsize=8, frameon=False)
    plt.tight_layout(); plt.show()
    if EQ_LEVELS is not None:
        print("note: EQ_LEVELS selects multigrid levels; the flat net has only level 0")

### 12d. Which term injects the stripes? (no weight changed)

One LPDS sweep is $x^+ = x - \tau r$ with

$$r = \underbrace{(E^HE\,x - \tilde y)}_{\text{data}} + \underbrace{D z}_{\text{synth}} - \pi_x,
\qquad \pi_x = r_c(\text{entry}) - R\,r_{\text{fine}},$$

and $z^+ = \operatorname{prox}(u)$, $u = z + A\bar x - \pi_z$. Every term of $r$ is linear, so the
near-Nyquist part of the step splits *exactly* into theirs (`trace_terms` hooks the sweeps; the
net runs unchanged). On a coarse level $-\pi_x$ is split in two: minus the level's own residual
where it entered, and plus the restricted fine residual.

Per sweep, on the complex field (not $|x|$, which makes harmonics), along rows and cols:
* **share** -- the term's projection onto the step's near-Nyquist part; the shares add to 100%,
  a negative one cancels others. **size** -- the term's near-Nyquist part over the step's: two
  big terms of opposite share are cancelling.
* **grouped** -- at a level's first sweep the state *is* $(Rx, Rz)$, so data + synth
  $- r_c(\text{entry})$ cancels up to how far training moved the smoother away from the FAS
  downsampler's copies; this row is that drift, against what the fine level hands down.
* **dual: D(piece of u)** -- the image's Nyquist band is reachable from the code only through
  $D$'s $W(\pi)$ (latent near-DC $\to$ image period 2; 12b showed that path is the one at L1),
  so this asks which piece of the dual argument carries the latent DC that $D$ leaks: the code
  $z$ itself, the fresh $A\bar x$, or the FAS correction $\pi_z$ (before the clip, the one
  nonlinearity). A large $\pi_z$ share points at the dual FAS placement (`fas_dual`); a large
  $z$ share at the code's own smooth content.

In [ ]:
# 12d. Split each sweep into its terms (the net runs unchanged)
LEVELS_12D = [1]          # grid levels to trace (0 = finest)
OUTERS_12D = None         # None: the last V-cycle; e.g. [0, -1]
SHOW_12D   = 0            # which recorded sweep to draw
FRAC = globals().get("FRAC", 0.8)

if NPZ:
    print("live mode only (NPZ is set)")
else:
    recs, m12 = lf.trace_terms(RUN, VOLUME, SLICE, levels=LEVELS_12D, outers=OUTERS_12D,
                               sigma=SIGMA, val_kspace_type=VAL_KSPACE, coarse_op=COARSE_OP,
                               transfer_filter=TRANSFER_FILTER)

    def _table(parts, title):
        att = lf.term_attribution(parts, FRAC)
        print(f"    {title:<30}{'rows: share':>13}{'size':>7}{'cols: share':>14}{'size':>7}")
        for (n, sr, zr), (_, sc, zc) in zip(att["rows"], att["cols"]):
            print(f"    {n:<30}{100 * sr:12.1f}%{zr:7.2f}{100 * sc:13.1f}%{zc:7.2f}")

    print(f"{m12['run']}: {m12['n_sweeps']} sweeps on levels {m12['levels']}, V-cycle(s) "
          f"{[o + 1 for o in m12['outers']] if m12['outers'] else 'all'}")
    for i, rec in enumerate(recs):
        (br, bc), (ar, ac) = lf.field_share(rec["x_before"], FRAC), lf.field_share(rec["x_after"], FRAC)
        print(f"\n[{i}] {rec['where']}  (fas_dual={rec['fas_dual']})   x near Nyquist: "
              f"rows {100 * br:.2f}% -> {100 * ar:.2f}%   cols {100 * bc:.2f}% -> {100 * ac:.2f}%")
        T = rec["terms"]
        _table(T, "primal residual r")
        if "FAS   -r_c(entry)" in T:
            own = T["data  E^H E x - y~"] + T["synth D z"] + T["FAS   -r_c(entry)"]
            _table({"own residual - r_c(entry)": own, "FAS   +R r_fine": T["FAS   +R r_fine"]},
                   "grouped")
        _table(rec["dual"], "dual: D(piece of u)")

    if recs:
        rec = recs[SHOW_12D]
        parts = [lf.nyquist_part(t, FRAC, "rows") for t in rec["terms"].values()]
        parts.append(lf.nyquist_part(sum(rec["terms"].values()), FRAC, "rows"))
        subplot_images([F(np.abs(p)) for p in parts], titles=list(rec["terms"]) + ["r (the sum)"],
                       share_window=True,
                       suptitle=f"{rec['where']}: each term's part at >= {FRAC:g} Nyquist "
                                f"along rows (one window)")